# Missing Value Imputation Demonstration In Python
##### Using Bank Loan Data

## **Objective:**
Build and compare multiple missing value imputation techniques for a bank loan dataset and demonstrate how different imputation strategies handle incomplete financial information before modeling.

## Background :
In real-world banking and credit risk data, missing values are common due to incomplete applications, data entry issues, or unavailable customer information. Handling these missing values appropriately is critical, as improper treatment can lead to biased estimates, reduced model performance, and incorrect credit decisions. Before building any predictive model, it is therefore essential to explore and apply suitable imputation techniques that preserve the underlying data structure and relationships between variables.

### Methods Demonstrated

In this exercise, we demonstrate and compare the following missing value imputation techniques:

**Simple Imputation**
Missing values are replaced using basic statistical measures such as mean or median. This method is easy to implement and serves as a baseline approach.

**Simple Imputation with Grouping**
Missing values are imputed using group-specific statistics (e.g., mean within defaulter or employment groups). This approach accounts for heterogeneity across different customer segments.

**K-Nearest Neighbors (KNN) Imputation**
Missing values are estimated based on the values of similar observations, identified using distance metrics. This method leverages local patterns in the data.

**Linear Regression Imputation**
A regression model is trained using complete cases to predict missing values based on relationships with other variables.

**Random Forest Imputation**
An ensemble-based approach that captures nonlinear relationships and interactions among variables, often providing more robust imputations for complex financial data.



## Data Description


| Variable   | Type      | Description                                                   |
|------------|-----------|---------------------------------------------------------------|
| `SN`       | Integer   | Serial number / ID of the applicant (identifier only)        |
| `AGE`      | Numeric   | Age band of the applicant (coded 1–6)                        |
| `EMPLOY`   | Numeric   | Years of employment / employment band                        |
| `ADDRESS`  | Numeric   | Years at current address (stability of residence)            |
| `DEBTINC`  | Numeric   | Debt-to-income ratio (%)                                     |
| `CREDDEBT` | Numeric   | Credit card debt                                             |
| `OTHDEBT`  | Numeric   | Other debts (e.g., personal loans)                           |
| `DEFAULTER`| Binary    | Target: 1 = default, 0 = non-default                         |

### Import Libraries

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

import warnings
warnings.filterwarnings("ignore")


### Import Data

In [2]:
df = pd.read_csv('BANKLOAN_MI.csv')
df.head()

,SN,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
0,1,3,17.0,12,9.3,11.36,5.01,1
1,2,1,10.0,6,NaN,1.36,4.00,0
2,3,2,15.0,14,5.5,0.86,2.17,0
3,4,3,15.0,14,2.9,2.66,0.82,0
4,5,1,2.0,0,17.3,1.79,3.06,1


In [3]:
# Count missing values column-wise
df.isna().sum()

SN           0
AGE          0
EMPLOY       2
ADDRESS      0
DEBTINC      3
CREDDEBT     0
OTHDEBT      0
DEFAULTER    0
dtype: int64

In [4]:
# Rows containing missing values
df[df.isna().any(axis=1)]

,SN,AGE,EMPLOY,ADDRESS,DEBTINC,CREDDEBT,OTHDEBT,DEFAULTER
1,2,1,10.0,6,NaN,1.36,4.00,0
6,7,2,NaN,9,30.6,3.83,16.67,0
7,8,3,12.0,11,NaN,0.13,1.24,0
14,15,3,22.0,15,NaN,3.70,5.40,0
22,23,1,NaN,6,10.0,0.43,2.17,0


In [5]:
# Store original data before imputation
df_before = df.copy()


# Columns with missing values
missing_cols = ['EMPLOY', 'DEBTINC']


### Method 1 : Simple Imputation Using Mean Value

In [6]:
df_impute = df.copy()
# Initialize Simple Imputer (Mean strategy)
simple_imputer = SimpleImputer(strategy='mean')

# Apply imputation
df_impute[missing_cols] = simple_imputer.fit_transform(df_impute[missing_cols])

# Verify missing values after imputation
df_impute[missing_cols].isna().sum()

print("Values BEFORE imputation:")
print(
    df_before[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)

print("\nValues AFTER imputation:")
print(
    df_impute[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)


Values BEFORE imputation:
    EMPLOY  DEBTINC
1     10.0      NaN
6      NaN     30.6
7     12.0      NaN
14    22.0      NaN
22     NaN     10.0

Values AFTER imputation:
       EMPLOY    DEBTINC
1   10.000000  10.261693
6    8.379656  30.600000
7   12.000000  10.261693
14  22.000000  10.261693
22   8.379656  10.000000


### Method 2 : Simple Imputation Using Group Mean

In [7]:
df_group = df.copy()
#Convert age into categprical variable
df_group['AGE'] = df_group['AGE'].astype('category')
# Impute missing values using AGE-wise mean
df_group[missing_cols] = (
    df_group.groupby('AGE')[missing_cols]
      .transform(lambda x: x.fillna(x.mean()))
)

# Verify missing values after imputation
df_group[missing_cols].isna().sum()

print("Values BEFORE group-wise imputation (by AGE):")
print(
    df_before[df_before[missing_cols].isna().any(axis=1)][['AGE'] + missing_cols]
)

print("\nValues AFTER group-wise imputation (by AGE):")
print(
    df_group[df_before[missing_cols].isna().any(axis=1)][['AGE'] + missing_cols]
)


Values BEFORE group-wise imputation (by AGE):
    AGE  EMPLOY  DEBTINC
1     1    10.0      NaN
6     2     NaN     30.6
7     3    12.0      NaN
14    3    22.0      NaN
22    1     NaN     10.0

Values AFTER group-wise imputation (by AGE):
   AGE     EMPLOY    DEBTINC
1    1  10.000000  10.299585
6    2   9.233216  30.600000
7    3  12.000000  10.350581
14   3  22.000000  10.350581
22   1   4.091286  10.000000


### Method 3: KNN Imputation

In [8]:
df_knn = df.copy()

# Initialize KNN Imputer
knn_imputer = KNNImputer(n_neighbors=5)

# Apply imputation
df_knn[missing_cols] = knn_imputer.fit_transform(df_knn[missing_cols])

# Verify missing values after imputation
df_knn[missing_cols].isna().sum()

print("Values BEFORE imputation:")
print(
    df_before[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)

print("\nValues AFTER KNN imputation:")
print(
    df_knn[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)


Values BEFORE imputation:
    EMPLOY  DEBTINC
1     10.0      NaN
6      NaN     30.6
7     12.0      NaN
14    22.0      NaN
22     NaN     10.0

Values AFTER KNN imputation:
    EMPLOY  DEBTINC
1     10.0    14.60
6      7.8    30.60
7     12.0    14.44
14    22.0    12.48
22    10.0    10.00


### Note :

KNN imputation treats all variables as numeric and computes distances between observations.
Categorical variables must be encoded numerically and are implicitly assumed to have an order.
This can lead to invalid category values, so KNN is generally better suited for continuous features.

### Method 4: Model Based Imputation Using Linear Regression

In [9]:
df_lr = df.copy()

# Initialize Iterative Imputer with Linear Regression
lr_imputer = IterativeImputer(estimator=LinearRegression(),max_iter=10,random_state=42)

# Apply imputation only on required columns
df_lr[missing_cols] = lr_imputer.fit_transform(df_lr[missing_cols])

# Verify missing values after imputation
df_lr[missing_cols].isna().sum()

print("Values BEFORE imputation:")
print(
    df_before[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)

print("\nValues AFTER Linear Regression (IterativeImputer) imputation:")
print(
    df_lr[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)


Values BEFORE imputation:
    EMPLOY  DEBTINC
1     10.0      NaN
6      NaN     30.6
7     12.0      NaN
14    22.0      NaN
22     NaN     10.0

Values AFTER Linear Regression (IterativeImputer) imputation:
       EMPLOY    DEBTINC
1   10.000000  10.196251
6    7.612481  30.600000
7   12.000000  10.116855
14  22.000000   9.719876
22   8.388389  10.000000


### Note :
Categorical variables must be encoded (e.g., one-hot or dummy encoding) before fitting the regression model. Linear regression does not natively understand categories; without proper encoding, categorical information cannot be used correctly for imputation.

### Method 5: Model Based Imputation Using Random Forest


In [10]:
df_rf = df.copy()

# Initialize Iterative Imputer with Random Forest
rf_estimator = RandomForestRegressor(n_estimators=100,random_state=42)

iter_imputer = IterativeImputer(estimator=rf_estimator,max_iter=10,random_state=42)

# Apply imputation (only on numeric columns)
df_rf[missing_cols] = iter_imputer.fit_transform(df_rf[missing_cols])

# Verify missing values after imputation
df_rf[missing_cols].isna().sum()

print("Values BEFORE imputation:")
print(
    df_before[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)

print("\nValues AFTER Iterative Random Forest imputation:")
print(
    df_rf[df_before[missing_cols].isna().any(axis=1)][missing_cols]
)


Values BEFORE imputation:
    EMPLOY  DEBTINC
1     10.0      NaN
6      NaN     30.6
7     12.0      NaN
14    22.0      NaN
22     NaN     10.0

Values AFTER Iterative Random Forest imputation:
       EMPLOY    DEBTINC
1   10.000000   9.745139
6    9.160000  30.600000
7   12.000000  12.388037
14  22.000000   9.199629
22  13.964167  10.000000
